In [2]:
%%writefile submission.py
# ！！！PLEASE DO NOT EDIT IT ！！！
# This command will save the code you wrote as a submission.py file in the same directory, for the test script to read.

# packages
from __future__ import
 annotations

import os
import sys
import importlib
import math
from collections.abc import Iterable
from typing import IO, Any, BinaryIO
import pickle

import numpy as np
import numpy.typing as npt
import torch
from jaxtyping import Bool, Float, Int
from torch import Tensor
import torch.nn.functional as F

# my answer functions
def run_linear(
    d_in: int,
    d_out: int,
    weights: Float[Tensor, " d_out d_in"],
    in_features: Float[Tensor, " ... d_in"],
) -> Float[Tensor, " ... d_out"]:
    return torch.matmul(in_features, weights.T)

def run_embedding(
    vocab_size: int,
    d_model: int,
    weights: Float[Tensor, " vocab_size d_model"],
    token_ids: Int[Tensor, " ..."],
) -> Float[Tensor, " ... d_model"]:
    return weights[token_ids]

def run_swiglu(
    d_model: int,
    d_ff: int,
    w1_weight: Float[Tensor, " d_ff d_model"],
    w2_weight: Float[Tensor, " d_model d_ff"],
    w3_weight: Float[Tensor, " d_ff d_model"],
    in_features: Float[Tensor, " ... d_model"],
) -> Float[Tensor, " ... d_model"]:
    
    gate = torch.silu(torch.matmul(in_features, w1_weight.T))  
    value = torch.matmul(in_features, w3_weight.T)
    gated = gate * value
    return torch.matmul(gated, w2_weight.T)

def run_rmsnorm(
    d_model: int,
    eps: float,
    weights: Float[Tensor, " d_model"],
    in_features: Float[Tensor, " ... d_model"],
) -> Float[Tensor, " ... d_model"]:
    
    variance = in_features.pow(2).mean(dim=-1, keepdim=True)
    normalized = in_features * torch.rsqrt(variance + eps)
    return normalized * weights

def run_silu(in_features: Float[Tensor, " ..."]) -> Float[Tensor, " ..."]:
    return torch.nn.functional.silu(in_features)

def run_scaled_dot_product_attention(
    Q: Float[Tensor, " ... queries d_k"],
    K: Float[Tensor, " ... keys d_k"],
    V: Float[Tensor, " ... values d_v"],
    mask: Bool[Tensor, " ... queries keys"] | None = None,
) -> Float[Tensor, " ... queries d_v"]:
    
    d_k = Q.size(-1)
    
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, -1e9)
    
    attn_weights = torch.softmax(scores, dim=-1)
    
    return torch.matmul(attn_weights, V)

def run_rope(
    d_k: int,
    theta: float,
    max_seq_len: int,
    in_query_or_key: Float[Tensor, " ... sequence_length d_k"],
    token_positions: Int[Tensor, " ... sequence_length"],
) -> Float[Tensor, " ... sequence_length d_k"]:

    batch_size, seq_len = in_query_or_key.shape[0], in_query_or_key.shape[1]
    
    x = in_query_or_key.view(batch_size, seq_len, -1, d_k)
    positions = token_positions.unsqueeze(-1).unsqueeze(-1)  # (batch_size, seq_len, 1, 1)
    
    dim = torch.arange(d_k // 2, device=x.device, dtype=torch.float32)
    dim = 10000.0 ** (-2 * dim / d_k)
    dim = positions * dim.unsqueeze(0).unsqueeze(0)  # (1, 1, d_k//2)
    
    cos = torch.cos(dim)
    sin = torch.sin(dim)
    
    x1 = x[..., 0::2] 
    x2 = x[..., 1::2]  
    
    rotated_x1 = x1 * cos - x2 * sin
    rotated_x2 = x1 * sin + x2 * cos

    result = torch.stack([rotated_x1, rotated_x2], dim=-1)
    result = result.view(batch_size, seq_len, -1)
    
    return result.view_as(in_query_or_key)

def run_multihead_self_attention(
    d_model: int,
    num_heads: int,
    q_proj_weight: Float[Tensor, " d_k d_in"],
    k_proj_weight: Float[Tensor, " d_k d_in"],
    v_proj_weight: Float[Tensor, " d_v d_in"],
    o_proj_weight: Float[Tensor, " d_model d_v"],
    in_features: Float[Tensor, " ... sequence_length d_in"],
) -> Float[Tensor, " ... sequence_length d_out"]:

    batch_size, seq_len, d_in = in_features.shape

    d_k = d_model // num_heads
    d_v = d_model // num_heads

    Q = torch.matmul(in_features, q_proj_weight.T)  # (batch_size, seq_len, d_model)
    K = torch.matmul(in_features, k_proj_weight.T)
    V = torch.matmul(in_features, v_proj_weight.T)
 
    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)  # (batch_size, num_heads, seq_len, d_k)
    K = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_heads, d_v).transpose(1, 2)

    d_k_value = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k_value)
    attn_weights = torch.softmax(scores, dim=-1)
    attn_output = torch.matmul(attn_weights, V)  # (batch_size, num_heads, seq_len, d_v)

    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

    return torch.matmul(attn_output, o_proj_weight.T)

def run_multihead_self_attention_with_rope(
    d_model: int,
    num_heads: int,
    max_seq_len: int,
    theta: float,
    q_proj_weight: Float[Tensor, " d_k d_in"],
    k_proj_weight: Float[Tensor, " d_k d_in"],
    v_proj_weight: Float[Tensor, " d_v d_in"],
    o_proj_weight: Float[Tensor, " d_model d_v"],
    in_features: Float[Tensor, " ... sequence_length d_in"],
    token_positions: Int[Tensor, " ... sequence_length"] | None = None,
) -> Float[Tensor, " ... sequence_length d_out"]:

    batch_size, seq_len, d_in = in_features.shape

    if token_positions is None:
        token_positions = torch.arange(seq_len, device=in_features.device).unsqueeze(0).expand(batch_size, seq_len)
    
    d_k = d_model // num_heads

    Q = torch.matmul(in_features, q_proj_weight.T)
    K = torch.matmul(in_features, k_proj_weight.T)
    V = torch.matmul(in_features, v_proj_weight.T)

    Q = run_rope(d_k, theta, max_seq_len, Q, token_positions)
    K = run_rope(d_k, theta, max_seq_len, K, token_positions)

    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    attn_weights = torch.softmax(scores, dim=-1)
    attn_output = torch.matmul(attn_weights, V)
    
    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
    
    return torch.matmul(attn_output, o_proj_weight.T)

def run_transformer_block(
    d_model: int,
    num_heads: int,
    d_ff: int,
    max_seq_len: int,
    theta: float,
    weights: dict[str, Tensor],
    in_features: Float[Tensor, " batch sequence_length d_model"],
) -> Float[Tensor, " batch sequence_length d_model"]:

    batch_size, seq_len, _ = in_features.shape
    
    residual = in_features
    x = run_rmsnorm(d_model, 1e-5, weights['ln1.weight'], in_features)
    
    if token_positions is None:
        token_positions = torch.arange(seq_len, device=in_features.device).unsqueeze(0).expand(batch_size, seq_len)
    
    attn_output = run_multihead_self_attention_with_rope(
        d_model=d_model,
        num_heads=num_heads,
        max_seq_len=max_seq_len,
        theta=theta,
        q_proj_weight=weights['attn.q_proj.weight'],
        k_proj_weight=weights['attn.k_proj.weight'],
        v_proj_weight=weights['attn.v_proj.weight'],
        o_proj_weight=weights['attn.output_proj.weight'],
        in_features=x,
        token_positions=token_positions
    )

    x = residual + attn_output
    
    residual = x
    x = run_rmsnorm(d_model, 1e-5, weights['ln2.weight'], x)
    
    ffn_output = run_swiglu(
        d_model=d_model,
        d_ff=d_ff,
        w1_weight=weights['ffn.w1.weight'],
        w2_weight=weights['ffn.w2.weight'],
        w3_weight=weights['ffn.w3.weight'],
        in_features=x
    )
    
    x = residual + ffn_output
    
    return x

def run_transformer_lm(
    vocab_size: int,
    context_length: int,
    d_model: int,
    num_layers: int,
    num_heads: int,
    d_ff: int,
    rope_theta: float,
    weights: dict[str, Tensor],
    in_indices: Int[Tensor, " batch_size sequence_length"],
) -> Float[Tensor, " batch_size sequence_length vocab_size"]:

    batch_size, seq_len = in_indices.shape

    x = run_embedding(vocab_size, d_model, weights['token_embeddings.weight'], in_indices)
    
    token_positions = torch.arange(seq_len, device=in_indices.device).unsqueeze(0).expand(batch_size, seq_len)

    for layer_idx in range(num_layers):
        layer_weights = {
            'attn.q_proj.weight': weights[f'layers.{layer_idx}.attn.q_proj.weight'],
            'attn.k_proj.weight': weights[f'layers.{layer_idx}.attn.k_proj.weight'],
            'attn.v_proj.weight': weights[f'layers.{layer_idx}.attn.v_proj.weight'],
            'attn.output_proj.weight': weights[f'layers.{layer_idx}.attn.output_proj.weight'],
            'ln1.weight': weights[f'layers.{layer_idx}.ln1.weight'],
            'ffn.w1.weight': weights[f'layers.{layer_idx}.ffn.w1.weight'],
            'ffn.w2.weight': weights[f'layers.{layer_idx}.ffn.w2.weight'],
            'ffn.w3.weight': weights[f'layers.{layer_idx}.ffn.w3.weight'],
            'ln2.weight': weights[f'layers.{layer_idx}.ln2.weight']
        }
        
        x = run_transformer_block(
            d_model=d_model,
            num_heads=num_heads,
            d_ff=d_ff,
            max_seq_len=context_length,
            theta=rope_theta,
            weights=layer_weights,
            in_features=x
        )
    
    x = run_rmsnorm(d_model, 1e-5, weights['ln_final.weight'], x)

    logits = torch.matmul(x, weights['lm_head.weight'].T)
    
    return logits

def run_get_batch(
    dataset: npt.NDArray, batch_size: int, context_length: int, device: str
) -> tuple[torch.Tensor, torch.Tensor]:

    if not isinstance(dataset, torch.Tensor):
        dataset = torch.from_numpy(dataset)

    starts = torch.randint(0, len(dataset) - context_length, (batch_size,))

    inputs = torch.stack([dataset[start:start+context_length] for start in starts])
    targets = torch.stack([dataset[start+1:start+context_length+1] for start in starts])
    
    return inputs.to(device), targets.to(device)

def run_softmax(in_features: Float[Tensor, " ..."], dim: int) -> Float[Tensor, " ..."]:

    return torch.softmax(in_features, dim=dim)

def run_cross_entropy(
    inputs: Float[Tensor, " batch_size vocab_size"], targets: Int[Tensor, " batch_size"]
) -> Float[Tensor, ""]:

    return torch.nn.functional.cross_entropy(inputs, targets)

def run_gradient_clipping(parameters: Iterable[torch.nn.Parameter], max_l2_norm: float) -> None:

    parameters = list(filter(lambda p: p.grad is not None, parameters))
    total_norm = torch.norm(torch.stack([torch.norm(p.grad.detach()) for p in parameters]), 2.0)
    
    if total_norm > max_l2_norm:
        clip_coef = max_l2_norm / (total_norm + 1e-6)
        for p in parameters:
            p.grad.detach().mul_(clip_coef)

def get_adamw_cls() -> Any:

    return torch.optim.AdamW

def run_get_lr_cosine_schedule(
    it: int,
    max_learning_rate: float,
    min_learning_rate: float,
    warmup_iters: int,
    cosine_cycle_iters: int,
):
    if it < warmup_iters:
        return max_learning_rate * (it / warmup_iters)
    
    progress = (it - warmup_iters) / cosine_cycle_iters
    cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
    
    return min_learning_rate + (max_learning_rate - min_learning_rate) * cosine_decay

def run_save_checkpoint(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    iteration: int,
    out: str | os.PathLike | BinaryIO | IO[bytes],
):

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'iteration': iteration
    }
    
    if isinstance(out, (str, os.PathLike)):
        torch.save(checkpoint, out)
    else:
        torch.save(checkpoint, out)

def run_load_checkpoint(
    src: str | os.PathLike | BinaryIO | IO[bytes],
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
) -> int:
    
    if isinstance(src, (str, os.PathLike)):
        checkpoint = torch.load(src, map_location='cpu')
    else:
        checkpoint = torch.load(src, map_location='cpu')
    
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    return checkpoint['iteration']

def get_tokenizer(
    vocab: dict[int, bytes],
    merges: list[tuple[bytes, bytes]],
    special_tokens: list[str] | None = None,
) -> Any:
    class SimpleBPETokenizer:
        def __init__(self, vocab, merges, special_tokens=None):
            self.vocab = vocab
            self.merges = merges
            self.special_tokens = special_tokens or []
            self.id_to_token = {idx: token for idx, token in enumerate(vocab)}
            self.token_to_id = {token.decode('utf-8', errors='replace'): idx for idx, token in vocab.items()}
    
    return SimpleBPETokenizer(vocab, merges, special_tokens)

def run_train_bpe(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
    **kwargs,
) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:

    vocab = {}
    merges = []
    
    with open(input_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    base_vocab = set(text)
    
    for idx, char in enumerate(base_vocab):
        vocab[idx] = char.encode('utf-8')

    return vocab, merges



Overwriting submission.py


In [3]:
import sys
import os
import pytest

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
os.chdir(project_root)

# Please CHANGE submission_dir to the directory of your answer.ipynb file, below is an example
submission_dir = os.path.join(project_root, "answers/yuanhao")
# submission_dir = os.path.join(project_root, "answers/xxxx")
if submission_dir not in sys.path:
    sys.path.insert(0, submission_dir)

# RUN TEST
# Args description:
# "-v": Verbose output
# "tests": Points to the test directory
# "-k sink": Run only tests related to "sink attention" (optional, for speed)
args = [
    "-v",
    "tests", 
    "-k", "test_attention_with_sink"
]

print(f"Current Working Directory: {os.getcwd()}")
pytest.main(args)

Current Working Directory: d:\
============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-9.0.2, pluggy-1.6.0 -- d:\juliannnnnn_project\venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: d:\
plugins: jaxtyping-0.3.6
collecting ... collected 0 items

============================ no tests ran in 0.00s ============================


ERROR: file or directory not found: tests



<ExitCode.USAGE_ERROR: 4>